### 1. RAG匯入準備工作

In [ ]:
# 下載檔案
!wget -O faiss_db.zip "https://drive.google.com/uc?export=download&id=1F2dkwwN9RU6UvF1OTx6zhkNsErN0wkWH"
!unzip faiss_db.zip

--2025-04-28 12:55:29--  https://drive.google.com/uc?export=download&id=1F2dkwwN9RU6UvF1OTx6zhkNsErN0wkWH
Resolving drive.google.com (drive.google.com)... 142.251.188.101, 142.251.188.139, 142.251.188.138, ...
Connecting to drive.google.com (drive.google.com)|142.251.188.101|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1F2dkwwN9RU6UvF1OTx6zhkNsErN0wkWH&export=download [following]
--2025-04-28 12:55:29--  https://drive.usercontent.google.com/download?id=1F2dkwwN9RU6UvF1OTx6zhkNsErN0wkWH&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.20.132, 2607:f8b0:400e:c07::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.20.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12420790 (12M) [application/octet-stream]
Saving to: ‘faiss_db.zip’

faiss_db.zip        100%[===================>]  11.84M  25.1MB/s

In [ ]:
!pip install -U langchain langchain-community sentence-transformers faiss-cpu gradio openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.2/661.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

In [ ]:
class CustomE5Embedding(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

In [ ]:
!ls -lh faiss_db

total 15M
-rw-r--r-- 1 root root  13M Apr 20 04:38 index.faiss
-rw-r--r-- 1 root root 2.4M Apr 20 04:38 index.pkl


In [ ]:
embedding_model = CustomE5Embedding(model_name="intfloat/multilingual-e5-small")
db = FAISS.load_local("faiss_db", embedding_model, allow_dangerous_deserialization=True)
retriever = db.as_retriever()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

### 2. 讀金鑰

In [ ]:
import os
from google.colab import userdata

In [ ]:
#【使用 OpenAI】
api_key = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY']=api_key
provider = "openai"
model = "gpt-4o"

In [ ]:
!pip install aisuite[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.7 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


### 3. 使用 AISuite 的準備

In [ ]:
import aisuite as ai

In [ ]:
provider_planner = "openai"
model_planner = "gpt-4o"

provider_writer = "openai"
model_writer = "gpt-4o"

provider_reviewer = "openai"
model_reviewer = "gpt-4o"

In [ ]:
def reply(system, prompt, provider="openai", model="gpt-4o"):
    """
    傳送 system prompt 和 user prompt，呼叫對應的模型提供回覆。
    """
    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]

    full_model_name = f"{provider}:{model}" if provider else model

    response = client.chat.completions.create(
        model=full_model_name,
        messages=messages
    )

    return response.choices[0].message.content.strip()


### 4. 打造二階段 + reflection

In [ ]:
system_planner = """
你是一位專業的 AI 學習規劃師，負責根據使用者的背景與目標，設計細緻且具邏輯性的學習路線。
請依照以下要求進行規劃：

- 條列出使用者目前的背景、基礎與限制條件。
- 明確定義使用者想達成的目標。
- 規劃短期（1-2個月）、中期（3-6個月）、長期（6個月以上）三個學習階段。
- 每個階段分別列出應學習的主題、需掌握的技能與具體學習任務。
- 保持內容條理清晰，結構分明，重點明確。

請使用繁體中文撰寫，語氣中性專業，避免使用感性或勵志語言。
"""

system_writer = """
你是一位專業的學習建議內容撰寫者，擅長整合學習規劃與補充資料，撰寫具結構性、實用性的學習建議文案。
請執行以下任務：

1. 依照短期、中期、長期的學習階段，從檢索結果中挑選適合該階段的書籍或資源，每個階段至少推薦一項。
2. 每本書或資源，請附上簡單說明，說明為何適合放在該階段閱讀或學習。
3. 接著，根據學習規劃與推薦書籍，撰寫一篇邏輯清楚、具實用性的學習建議文案，說明每個階段的學習重點、安排原因與應用場景。
4. 語氣請保持中性專業，使用繁體中文，不需加入激勵語氣或情緒化措辭。
5. 若某筆資料缺少書名或關鍵資訊，請略過該筆資料，避免出現「書名：無」或類似空白資訊。
6. 最後可補充提醒：「你可以到 https://z-lib.id/ 搜尋這些書籍」。

請將學習建議以結構清楚、條理分明的方式撰寫，讓使用者能夠理解每個學習階段的安排意義與學習資源的用途。
"""

system_reviewer = """
你是一位專業的學習諮詢師與內容審核專家，擅長檢視學習建議文案的邏輯性、條理性與專業性。
請你針對以下貼文，從以下幾個面向進行具體審閱與修改建議：
1. 是否符合使用者背景與目標
2. 各學習階段的安排是否合理、有連貫性
3. 說明是否清晰易懂、專業但不艱澀
4. 是否有缺漏、矛盾或可以補充之處

請具體指出需要修改或優化的地方，並提出改寫建議或補充說明。
回應請使用台灣習慣的中文，語氣正式、專業，像是一位專家正在進行審閱說明。
"""

In [ ]:
def rag_retrieve(query):
    """
    使用自己的 retriever，根據 query 檢索最相關的文件。
    """
    docs = retriever.get_relevant_documents(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
def advisor_post_full_with_rag(user_input):
    """
    完整版：
    - 推理規劃 (planner)
    - RAG檢索補充資料
    - 初版建議撰寫 (writer1)
    - 專業反思與修正建議 (reviewer)
    - 參考原規劃 + RAG + 反思後，重新優化撰寫 (writer2)
    """

    # Step 1: 學習規劃推理
    planning_prompt = f"使用者輸入：「{user_input}」。請規劃短期、中期與長期的學習路徑，條列清楚。"
    learning_plan = reply(
        system=system_planner,
        prompt=planning_prompt,
        provider=provider_planner,
        model=model_planner
    )

    # Step 2: RAG 資料檢索
    rag_query = learning_plan
    rag_retrieval_result = rag_retrieve(rag_query)

    # Step 3: 初版學習建議撰寫 (writer 第一次)
    generation_prompt = f"""請根據以下學習規劃與檢索到的資料，撰寫一篇完整的學習建議文案：
    【學習規劃】
    {learning_plan}

    【檢索資料】
    {rag_retrieval_result}
    """
    initial_post = reply(
        system=system_writer,
        prompt=generation_prompt,
        provider=provider_writer,
        model=model_writer
    )

    # Step 4: 專業反思 (reviewer)
    reflection_prompt = f"""請檢查以下學習建議文案是否有邏輯問題、說明不清或安排不合理之處，並提出具體修改建議：{initial_post}"""

    reflection_result = reply(
        system=system_reviewer,
        prompt=reflection_prompt,
        provider=provider_reviewer,
        model=model_reviewer
    )

    # Step 5: 優化版建議撰寫 (writer 第二次)【重點：重新結合 learning_plan + rag資料 + reflection修正意見】
    improvement_prompt = f"""以下是原本的學習建議文案：
    {initial_post}

    以下是專業反思提出的修正建議：
    {reflection_result}

    請參考以下內容，重新撰寫優化版的學習建議文案：
    【學習規劃】
    {learning_plan}

    【檢索資料】
    {rag_retrieval_result}

    【反思建議】
    {reflection_result}

    要求：
    - 保持原本學習規劃與資料結合的完整性
    - 根據反思提出的問題進行優化修正
    - 保持內容邏輯清晰、結構完整、繁體中文，中性專業語氣
    """
    improved_post = reply(
        system=system_writer,
        prompt=improvement_prompt,
        provider=provider_writer,
        model=model_writer
    )

    return learning_plan, initial_post, reflection_result, improved_post


### 5. 用 Gradio 打造對話機器人

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("🎓 AI 自主學習諮詢師(正式版)")
    user_input = gr.Textbox(
        label="請描述你的背景與想要達成的目標（例如：我想轉職做AI工程師）")
    btn = gr.Button("生成完整學習規劃建議")

    with gr.Row():
        out1 = gr.Textbox(label="規劃出的學習路徑（推理結果）", lines=10)
        out2 = gr.Textbox(label="初版學習建議文案（第一次撰寫）", lines=10)
        out3 = gr.Textbox(label="專業反思與修正建議（審閱結果）", lines=8)
        out4 = gr.Textbox(label="優化後最終版學習建議（重新撰寫）", lines=12)

    btn.click(
        fn=advisor_post_full_with_rag,
        inputs=[user_input],
        outputs=[out1, out2, out3, out4]
    )

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ff37032394dc0f4438.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<ipython-input-14-3c82fb91a09e>:5: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(query)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ff37032394dc0f4438.gradio.live
